# Phase 8 — Inference on New Images
**RealCentric Generator-Agnostic Deepfake Detection**  
SVKM AI/ML HPC Cluster

Run deepfake detection on **any new image**.  
Change the path in Step 3 and run all cells.

**Output per image:** Label (REAL/FAKE), confidence, heatmap, per-detector breakdown  
Project root: `/data/mpstme-naman/deepfake_detection`

## Step 1 — Load All Trained Models

In [ ]:
import sys, cv2, numpy as np, pickle
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path

BASE     = Path('/data/mpstme-naman/deepfake_detection')
CKPT_DIR = BASE / 'checkpoints'
PROC     = BASE / 'data' / 'processed'
RES_DIR  = BASE / 'results' / 'inference'; RES_DIR.mkdir(parents=True, exist_ok=True)

from config.config_loader          import load_config
from src.features.extractor        import FeatureFusionPipeline
from src.models.mlp_classifier     import MLPTrainer
from src.models.one_class_ensemble import OneClassEnsemble
from src.models.autoencoder        import DeepfakeExplainer

cfg = load_config()

pipeline = FeatureFusionPipeline(cfg=cfg, backbone='resnet18')
with open(CKPT_DIR/'pipeline_state.pkl','rb') as f: pipeline.set_state(pickle.load(f))
print('  ✓  Pipeline loaded')

mlp = MLPTrainer(cfg=cfg, input_dim=pipeline.output_dim)
mlp.load_checkpoint(str(CKPT_DIR/'mlp_supervised_best.pt'))
print('  ✓  MLP loaded')

ensemble = OneClassEnsemble(cfg=cfg)
ensemble.load(str(CKPT_DIR/'ensemble.pkl'))
print(f'  ✓  Ensemble loaded  τ={ensemble.threshold:.4f}')

explainer = DeepfakeExplainer(cfg=cfg)
try:
    explainer.load_autoencoder(str(CKPT_DIR/'autoencoder_best.pt'))
    print('  ✓  Autoencoder loaded')
except:
    print('  ○  Autoencoder not found — ELA-only mode')

print('\n  All models ready!')

## Step 2 — Define Inference Function

In [ ]:
import matplotlib.pyplot as plt

def predict_image(image_path, save=True):
    """
    Run full deepfake detection on one image.
    Returns result dict and shows a visual report.
    """
    img = cv2.imread(str(image_path))
    if img is None: print(f'ERROR: Cannot load {image_path}'); return None

    Z         = pipeline.extract(img, normalise=True).reshape(1, -1)
    mlp_prob  = float(mlp.predict_proba(Z)[0])
    mlp_label = 'FAKE' if mlp_prob >= 0.5 else 'REAL'
    occ_score = float(ensemble.score_features(Z)[0])
    occ_label = 'FAKE' if occ_score > ensemble.threshold else 'REAL'
    exp       = explainer.explain(img)

    final_label = 'FAKE' if (mlp_label=='FAKE' or occ_label=='FAKE') else 'REAL'
    colour      = '#d32f2f' if final_label == 'FAKE' else '#1976d2'

    fig = plt.figure(figsize=(14, 5))
    fig.suptitle(f'VERDICT: {final_label}  |  {Path(image_path).name}',
                 fontsize=15, fontweight='bold', color=colour)

    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax1.set_title('Input Image'); ax1.axis('off')

    ax2 = fig.add_subplot(1, 3, 2)
    im  = ax2.imshow(exp['heatmap'], cmap='jet', vmin=0, vmax=0.4)
    ax2.set_title('Anomaly Heatmap\n(bright = suspicious region)'); ax2.axis('off')
    plt.colorbar(im, ax=ax2, fraction=0.046)

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.imshow(cv2.cvtColor(exp['overlay'], cv2.COLOR_BGR2RGB)); ax3.set_title('Overlay'); ax3.axis('off')

    fig.text(0.5, -0.03,
             f'MLP:      {mlp_label}  (prob={mlp_prob:.3f})\n'
             f'Ensemble: {occ_label}  (score={occ_score:.3f}, τ={ensemble.threshold:.3f})\n'
             f'ELA:      score={exp["score"]:.4f}',
             ha='center', va='top', fontsize=10, family='monospace',
             bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.9))

    plt.tight_layout()
    if save:
        out = RES_DIR / f'{Path(image_path).stem}_result.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        print(f'  Report → {out}')
    plt.show()

    print(f'  Final    : {final_label}')
    print(f'  MLP      : {mlp_label}  prob={mlp_prob:.4f}')
    print(f'  Ensemble : {occ_label}  score={occ_score:.4f}  τ={ensemble.threshold:.4f}')
    print(f'  ELA      : score={exp["score"]:.4f}')
    return {'label':final_label,'mlp_prob':mlp_prob,'occ_score':occ_score,'ela_score':exp['score']}

print('  ✓  predict_image() ready')
print(f'  Usage: result = predict_image("{BASE}/data/processed/celebdf/fake/fake_0000001.png")')

## Step 3 — Run Detection on Your Image

⬇️ **Change this path to your image**

In [ ]:
# ── CHANGE THIS PATH to your image ───────────────────────────────────
IMAGE_PATH = '/data/mpstme-naman/deepfake_detection/data/processed/celebdf/fake/fake_0000001.png'
# ─────────────────────────────────────────────────────────────────────

result = predict_image(IMAGE_PATH)

## Step 4 — Batch Inference on Multiple Images

In [ ]:
import csv

def predict_batch(image_paths, save_csv=True):
    """Run detection on a list of images. Returns results list."""
    results = []
    print(f'Batch inference: {len(image_paths)} images\n')
    for i, path in enumerate(image_paths):
        img = cv2.imread(str(path))
        if img is None: continue
        Z   = pipeline.extract(img, normalise=True).reshape(1, -1)
        mp  = float(mlp.predict_proba(Z)[0])
        os_ = float(ensemble.score_features(Z)[0])
        es  = float(explainer._ela.score(img))
        lbl = 'FAKE' if (mp >= 0.5 or os_ > ensemble.threshold) else 'REAL'
        results.append({'file':Path(path).name,'verdict':lbl,
                        'mlp_prob':round(mp,4),'occ_score':round(os_,4),'ela_score':round(es,4)})
        print(f'  [{i+1:>4}/{len(image_paths)}]  {Path(path).name:<42}  {lbl}', end='\r')
    print()
    if save_csv and results:
        cp = RES_DIR/'batch_results.csv'
        with open(cp,'w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=results[0].keys()); w.writeheader(); w.writerows(results)
        print(f'  ✓  CSV → {cp}')
    fake_n = sum(1 for r in results if r['verdict']=='FAKE')
    print(f'\n  {fake_n}/{len(results)} images detected as FAKE')
    return results

# Example: 10 real + 10 fake from CelebDF
test_paths = (sorted((PROC/'celebdf'/'real').glob('*.png'))[:10] +
              sorted((PROC/'celebdf'/'fake').glob('*.png'))[:10])
batch_res  = predict_batch(test_paths)

print()
print(f'  {"File":<40} {"Verdict":<8} {"MLP":>7} {"Ensemble":>10}')
print('  '+'-'*68)
for r in batch_res:
    print(f'  {r["file"]:<40} {r["verdict"]:<8} {r["mlp_prob"]:>7.4f} {r["occ_score"]:>10.4f}')

## Step 5 — Test on Stable Diffusion (Unseen Generator)

All SD images are AI-generated fakes from a generator the model **never saw during training**.

In [ ]:
sd_paths = sorted((PROC/'sd'/'fake').glob('*.png'))[:30]
sd_res   = predict_batch(sd_paths, save_csv=False)
detected = sum(1 for r in sd_res if r['verdict']=='FAKE')
print(f'\n  Stable Diffusion detection: {detected}/{len(sd_res)} = {detected/len(sd_res)*100:.1f}%')
print('  (Higher = better — all are genuine AI fakes)')

## ✅ All 8 Phases Complete

**Your deepfake detector is fully operational.**

```python
# Detect any single image:
result = predict_image('/path/to/image.jpg')

# Detect a folder:
paths   = list(Path('/your/folder').glob('*.jpg'))
results = predict_batch(paths)
```

All results saved to:  
`/data/mpstme-naman/deepfake_detection/results/inference/`